# Epic 1 Demo — The Foundation Layer

**What is this?** ReformLab is a platform for analysing the impact of environmental policies (like a carbon tax) on households. Before we can run any analysis, we need solid plumbing: a way to load data, talk to the tax calculator, check that nothing is broken, and keep track of where every number came from.

That plumbing is what Epic 1 built. This notebook walks through it step by step:

1. **Loading data files** — reading household and emission data, catching errors early
2. **Tracking where data comes from** — fingerprinting every file so results are reproducible
3. **Translating column names** — the tax calculator uses its own naming; we map to ours
4. **Running the tax calculator** — via pre-computed files or a fake stand-in for testing
5. **Checking output quality** — catching missing values, wrong types, out-of-range numbers
6. **Version compatibility** — knowing which versions of the tax calculator we support

In [3]:
from pathlib import Path

# Point to the folder with our sample data files
NOTEBOOK_DIR = Path.cwd() if "__file__" not in dir() else Path("__file__").parent
DATA_DIR = NOTEBOOK_DIR / "data"
print(f"Data directory: {DATA_DIR}")
print(f"Files: {sorted(p.name for p in DATA_DIR.iterdir())}")


def show(table, n=8):
    """Helper to display a data table as a readable text table."""
    sliced = table.slice(0, n) if table.num_rows > n else table
    cols = sliced.column_names
    rows = [
        [str(sliced.column(c)[i].as_py()) for c in cols]
        for i in range(sliced.num_rows)
    ]
    widths = [max(len(c), *(len(r[j]) for r in rows)) for j, c in enumerate(cols)]
    header = "  ".join(c.ljust(w) for c, w in zip(cols, widths))
    sep = "  ".join("-" * w for w in widths)
    print(header)
    print(sep)
    for row in rows:
        print("  ".join(v.ljust(w) for v, w in zip(row, widths)))
    if table.num_rows > n:
        print(f"  ... ({table.num_rows - n} more rows)")

Data directory: /Users/lucas/Workspace/reformlab/notebooks/data
Files: ['emission_factors.csv', 'field_mapping.yaml', 'openfisca_output_2024.csv', 'synthetic_population.csv']


---
## 1. Loading a Data File

Our analyses start with data about people and households (age, income, region, etc.). That data lives in CSV files — simple spreadsheets.

When we load a file, the system checks that it has the right columns and the right data types **before** anything else happens. Think of it like a bouncer at the door: if your file is missing a required column, it gets rejected immediately with a clear explanation of what's wrong.

In [4]:
from reformlab.computation import ingest_csv
from reformlab.data import SYNTHETIC_POPULATION_SCHEMA

# Load the CSV and check it against the expected schema (column names + types)
pop_result = ingest_csv(DATA_DIR / "synthetic_population.csv", SYNTHETIC_POPULATION_SCHEMA)

print(f"Format:    {pop_result.format}")
print(f"Rows:      {pop_result.row_count}")
print(f"Source:    {pop_result.source_path.name}")
print(f"Loaded at: {pop_result.metadata['loaded_at']}")
print()
print("Schema (what each column is called and its data type):")
print(pop_result.table.schema)
print()
show(pop_result.table)

Format:    csv
Rows:      12
Source:    synthetic_population.csv
Loaded at: 2026-02-26T15:12:01+00:00

Schema (what each column is called and its data type):
household_id: int64
person_id: int64
age: int64
income: double
region_code: string
housing_status: string
household_size: int64

household_id  person_id  age  income   region_code  housing_status  household_size
------------  ---------  ---  -------  -----------  --------------  --------------
1             101        42   32000.0  11           owner           3             
1             102        39   28000.0  11           owner           3             
1             103        12   0.0      11           owner           3             
2             201        67   18500.0  75           tenant          1             
3             301        28   24000.0  33           tenant          2             
3             302        26   22000.0  33           tenant          2             
4             401        55   45000.0  69        

### What happens when the data is wrong?

Below we deliberately feed in a file that's missing two required columns (`age` and `income`). Instead of crashing mysteriously, the system tells you exactly which columns are missing and what to do about it.

In [5]:
import tempfile, csv
from reformlab.computation import IngestionError

# Write a CSV missing the required "age" and "income" columns
bad_csv = Path(tempfile.mktemp(suffix=".csv"))
bad_csv.write_text("household_id,person_id\n1,101\n")

try:
    ingest_csv(bad_csv, SYNTHETIC_POPULATION_SCHEMA)
except IngestionError as e:
    print(f"Caught IngestionError:")
    print(f"  Missing columns: {e.missing_columns}")
    print(f"  Message: {e}")
finally:
    bad_csv.unlink(missing_ok=True)

Caught IngestionError:
  Missing columns: ('age', 'income')
  Message: Ingestion failed - missing required columns: age, income - Add all required columns to the source file and retry (file: /var/folders/p9/3r4_fgzd72j7b469xxshgfnh0000gn/T/tmpfud0pxkv.csv)


---
## 2. Tracking Where Data Comes From (Provenance)

For any policy analysis to be trusted, you need to be able to answer: *"Where did this data come from? Is it the same file we used last time?"*

Every time we load a dataset, the system:
- Computes a **fingerprint** (SHA-256 hash) of the file — like a unique ID that changes if even one number changes
- Records **who published it**, what version it is, and when we loaded it
- Stores all of this in a **registry** so we can reconstruct any analysis later

This is what makes results **reproducible**: anyone can check that they're using the exact same input data.

In [6]:
from reformlab.data import (
    DataSourceMetadata,
    DatasetRegistry,
    EMISSION_FACTOR_SCHEMA,
    load_dataset,
    build_emission_factor_index,
)

# Describe where this dataset comes from (name, version, URL, license)
pop_source = DataSourceMetadata(
    name="synthetic_population_fr",
    version="2024.1",
    url="https://example.com/synthetic-pop",
    description="Synthetic French households (demo)",
    license="Open Data Commons",
)

# Load the file — this validates the data AND computes a fingerprint
pop_table, pop_manifest = load_dataset(
    DATA_DIR / "synthetic_population.csv",
    SYNTHETIC_POPULATION_SCHEMA,
    pop_source,
)

print("=== Population Manifest (the dataset's ID card) ===")
print(f"  Key:         {pop_manifest.dataset_key}")
print(f"  SHA-256:     {pop_manifest.content_hash[:24]}...")
print(f"  Rows:        {pop_manifest.row_count}")
print(f"  Columns:     {pop_manifest.column_names}")
print(f"  Loaded at:   {pop_manifest.loaded_at}")

=== Population Manifest (the dataset's ID card) ===
  Key:         synthetic_population_fr@2024.1:197cc62f8795
  SHA-256:     197cc62f87957f2cffbc72b9...
  Rows:        12
  Columns:     ('household_id', 'person_id', 'age', 'income', 'region_code', 'housing_status', 'household_size')
  Loaded at:   2026-02-26T15:12:01+00:00


In [7]:
# Load emission factors (how much CO2 per unit of activity)
ef_source = DataSourceMetadata(
    name="emission_factors_ademe",
    version="2024.1",
    url="https://example.com/emission-factors",
    description="ADEME/Agribalyse emission factors (demo)",
    license="Etalab Open License",
)

ef_table, ef_manifest = load_dataset(
    DATA_DIR / "emission_factors.csv",
    EMISSION_FACTOR_SCHEMA,
    ef_source,
)

# Build a lookup index so we can query by category and year
ef_index = build_emission_factor_index(ef_table)

print("=== Emission Factor Index ===")
print(f"  Categories:  {ef_index.categories()}")
print()
print("Transport factors for 2024 (kgCO2 per km):")
show(ef_index.by_category_and_year("transport", 2024))

=== Emission Factor Index ===
  Categories:  ('food', 'housing', 'transport')

Transport factors for 2024 (kgCO2 per km):
category   factor_value  unit      year  subcategory  source  co2_equivalent
---------  ------------  --------  ----  -----------  ------  --------------
transport  0.218         kgCO2/km  2024  car_petrol   ADEME   0.218         
transport  0.195         kgCO2/km  2024  car_diesel   ADEME   0.195         
transport  0.025         kgCO2/km  2024  train        ADEME   0.025         


In [8]:
# The registry keeps a record of every dataset we've loaded
# This is the audit trail — useful for governance and reproducibility
registry = DatasetRegistry()
registry.register(pop_manifest)
registry.register(ef_manifest)

print("Registry contents (can be saved as JSON for audit trails):")
import json
print(json.dumps(registry.to_dict(), indent=2, default=str))

Registry contents (can be saved as JSON for audit trails):
{
  "synthetic_population_fr@2024.1:197cc62f8795": {
    "source": {
      "name": "synthetic_population_fr",
      "version": "2024.1",
      "url": "https://example.com/synthetic-pop",
      "description": "Synthetic French households (demo)",
      "license": "Open Data Commons"
    },
    "content_hash": "197cc62f87957f2cffbc72b9c248ae2b7739cf0048a4a646390ca367b1adf19d",
    "file_path": "/Users/lucas/Workspace/reformlab/notebooks/data/synthetic_population.csv",
    "format": "csv",
    "row_count": 12,
    "column_names": [
      "household_id",
      "person_id",
      "age",
      "income",
      "region_code",
      "housing_status",
      "household_size"
    ],
    "loaded_at": "2026-02-26T15:12:01+00:00"
  },
  "emission_factors_ademe@2024.1:dbe7703a8a87": {
    "source": {
      "name": "emission_factors_ademe",
      "version": "2024.1",
      "url": "https://example.com/emission-factors",
      "description": "ADE

---
## 3. Translating Column Names (Field Mapping)

The tax calculator (OpenFisca) uses its own column names — for example `income_tax`. But in our project we might want to use French names like `impot_revenu`, or just different conventions.

Instead of hard-coding these translations everywhere, we define them once in a simple YAML configuration file. The system then automatically renames columns in both directions:
- **Going in**: our names to OpenFisca names
- **Coming out**: OpenFisca names back to ours

It also checks that every name in the mapping actually exists in the data — and if you make a typo, it suggests the closest match.

In [9]:
from reformlab.computation import load_mapping, validate_mapping

# Load the YAML file that defines our column name translations
mapping = load_mapping(DATA_DIR / "field_mapping.yaml")

print("Loaded mappings (OpenFisca name <-> our project name):")
for fm in mapping.mappings:
    print(f"  {fm.openfisca_name:15s} <-> {fm.project_name:15s}  [{fm.direction:6s}]")

print(f"\nColumns we rename on output: {[m.openfisca_name for m in mapping.output_mappings]}")
print(f"Columns we rename on input:  {[m.openfisca_name for m in mapping.input_mappings]}")

Loaded mappings (OpenFisca name <-> our project name):
  income_tax      <-> impot_revenu     [output]
  carbon_tax      <-> taxe_carbone     [output]
  household_id    <-> id_menage        [both  ]
  person_id       <-> id_personne      [both  ]

Columns we rename on output: ['income_tax', 'carbon_tax', 'household_id', 'person_id']
Columns we rename on input:  ['household_id', 'person_id']


In [10]:
# Check that every name in our mapping actually exists in the data
available_cols = ("household_id", "person_id", "income_tax", "carbon_tax")
result = validate_mapping(mapping, available_cols)
print(f"Validation errors:   {result.errors}")
print(f"Validation warnings: {result.warnings}")
print("All good — every mapped name was found in the data!")

Validation errors:   ()
Validation warnings: ()
All good — every mapped name was found in the data!


---
## 4. Running the Tax Calculator

This is the core of the system: given a population and a policy, compute the taxes.

We don't build our own tax engine — we use **OpenFisca**, an open-source tax-benefit calculator maintained by the French government. Our code talks to it through an **adapter** (a standardised plug), which means we could swap in a different calculator without rewriting everything else.

### 4a. File-based mode

The simplest mode: OpenFisca has already been run separately, and we just read the result files. This is fast and doesn't require OpenFisca to be installed.

Below we load a pre-computed CSV, then apply our column name mapping to translate the output into our project's naming convention.

In [11]:
from reformlab.computation.openfisca_adapter import OpenFiscaAdapter
from reformlab.computation import PopulationData, PolicyConfig, apply_output_mapping

# Set up a folder with a pre-computed result file (named by year)
import shutil
adapter_dir = DATA_DIR / "adapter_store"
adapter_dir.mkdir(exist_ok=True)
shutil.copy(DATA_DIR / "openfisca_output_2024.csv", adapter_dir / "2024.csv")

# Create the adapter (skip_version_check=True because OpenFisca isn't installed here)
adapter = OpenFiscaAdapter(adapter_dir, skip_version_check=True)

# Describe our inputs: who are the people, and what policy are we testing?
population = PopulationData(
    tables={"default": pop_table},
    metadata={"source": pop_manifest.dataset_key},
)
policy = PolicyConfig(
    parameters={"carbon_tax_rate": 44.6},
    name="baseline_2024",
    description="Baseline carbon tax scenario at 44.6 EUR/tCO2",
)

# "Compute" — in file mode this just reads the pre-computed CSV for year 2024
result = adapter.compute(population, policy, period=2024)

print(f"Adapter version: {adapter.version()}")
print(f"Period:          {result.period}")
print(f"Output rows:     {result.output_fields.num_rows}")
print()
print("Raw output from the tax calculator:")
show(result.output_fields)

Adapter version: unknown
Period:          2024
Output rows:     12

Raw output from the tax calculator:
household_id  person_id  income_tax  carbon_tax
------------  ---------  ----------  ----------
1             101        4800.0      156.0     
1             102        4200.0      156.0     
1             103        0.0         156.0     
2             201        1110.0      210.0     
3             301        3600.0      132.0     
3             302        3300.0      132.0     
4             401        9000.0      178.0     
4             402        7600.0      178.0     
  ... (4 more rows)


In [12]:
# Now translate the column names: income_tax -> impot_revenu, etc.
mapped_table = apply_output_mapping(result.output_fields, mapping)

print("Same data, but with our project's column names:")
show(mapped_table)

Same data, but with our project's column names:
id_menage  id_personne  impot_revenu  taxe_carbone
---------  -----------  ------------  ------------
1          101          4800.0        156.0       
1          102          4200.0        156.0       
1          103          0.0           156.0       
2          201          1110.0        210.0       
3          301          3600.0        132.0       
3          302          3300.0        132.0       
4          401          9000.0        178.0       
4          402          7600.0        178.0       
  ... (4 more rows)


### 4b. Fake stand-in for testing (Mock Adapter)

When developing the rest of the platform (scenario orchestration, indicators, etc.), we don't want to depend on OpenFisca being installed. So we built a "mock" adapter — a fake calculator that returns whatever numbers we configure.

It looks and behaves exactly like the real adapter from the outside (it satisfies the same interface contract), but it has zero dependencies and runs instantly. It also keeps a log of every call, which is useful for testing.

In [13]:
import pyarrow as pa
from reformlab.computation import ComputationAdapter
from reformlab.computation.mock_adapter import MockAdapter

# Create a fake adapter with hand-picked output numbers
mock_output = pa.table({
    "income_tax": pa.array([3200.0, 1800.0, 5400.0]),
    "carbon_tax": pa.array([150.0, 210.0, 170.0]),
})
mock = MockAdapter(version_string="mock-carbon-v1", default_output=mock_output)

# The mock "quacks like" the real adapter — it satisfies the same contract
print(f"Satisfies the ComputationAdapter contract: {isinstance(mock, ComputationAdapter)}")

# Run it — returns our hand-picked numbers regardless of the input
mock_result = mock.compute(population, policy, period=2024)
print(f"Mock version: {mock.version()}")
print(f"Mock output:")
show(mock_result.output_fields)

# The mock also logs every call — useful for verifying test behaviour
print(f"\nCall log: {mock.call_log}")

Satisfies the ComputationAdapter contract: True
Mock version: mock-carbon-v1
Mock output:
income_tax  carbon_tax
----------  ----------
3200.0      150.0     
1800.0      210.0     
5400.0      170.0     

Call log: [{'population_row_count': 12, 'policy_name': 'baseline_2024', 'period': 2024}]


---
## 5. Checking Output Quality

Even if the data loads fine and the calculator runs, we still need to check the **results** before using them. Things that could go wrong:
- A column that should always have a value has blanks (nulls)
- A column has the wrong data type (text where we expected numbers)
- Values are outside a reasonable range (negative tax? a million-euro carbon tax on one person?)

The first cell below runs quality checks on valid output — everything passes. The second cell deliberately injects missing values to show what a failure looks like: the system identifies exactly which fields have problems and in which rows.

In [14]:
from reformlab.computation import (
    validate_output,
    DEFAULT_OPENFISCA_OUTPUT_SCHEMA,
    RangeRule,
    DataQualityError,
)

# Run quality checks on the real result — should pass with flying colours
qc = validate_output(
    result,
    DEFAULT_OPENFISCA_OUTPUT_SCHEMA,
    range_rules=(
        # We also define "reasonable" bounds — values outside these trigger a warning
        RangeRule(field="income_tax", min_value=0, max_value=100_000, description="Reasonable income tax range"),
        RangeRule(field="carbon_tax", min_value=0, max_value=5_000, description="Reasonable carbon tax range"),
    ),
)

print(f"Quality check passed: {qc.passed}")
print(f"Checked columns:      {qc.checked_columns}")
print(f"Rows validated:       {qc.row_count}")
print(f"Errors:               {len(qc.errors)}")
print(f"Warnings:             {len(qc.warnings)}")

Quality check passed: True
Checked columns:      ('income_tax', 'carbon_tax', 'household_id', 'person_id')
Rows validated:       12
Errors:               0
Warnings:             0


In [15]:
# Deliberately create bad data with missing values (None = blank)
from reformlab.computation import ComputationResult

bad_table = pa.table({
    "income_tax": pa.array([1200.0, None, 3400.0]),   # row 1 is blank!
    "carbon_tax": pa.array([100.0, 200.0, None]),      # row 2 is blank!
})
bad_result = ComputationResult(
    output_fields=bad_table,
    adapter_version="test",
    period=2024,
)

# The quality check catches the blanks and tells us exactly where they are
try:
    validate_output(bad_result, DEFAULT_OPENFISCA_OUTPUT_SCHEMA)
except DataQualityError as e:
    print("Quality validation FAILED (as expected):")
    print(f"  Blocking errors: {len(e.result.errors)}")
    for issue in e.result.errors:
        print(f"    Column '{issue.field}': {issue.issue_type} at row(s) {issue.row_indices}")

Quality validation FAILED (as expected):
  Blocking errors: 2
    Column 'income_tax': null_values at row(s) (1,)
    Column 'carbon_tax': null_values at row(s) (2,)


---
## 6. Version Compatibility

OpenFisca evolves over time — new versions may change how calculations work. We maintain a compatibility matrix that tracks which versions we've tested against.

For any version, the system tells you:
- **Supported** — tested and known to work
- **Untested** — might work but we haven't verified it
- **Unsupported** — too old or not a real version number

In [16]:
from reformlab.computation import get_compatibility_info

# Check a few versions — supported, untested, too old, and invalid
for version in ["44.0.0", "44.2.2", "45.0.0", "43.0.0", "not-installed"]:
    info = get_compatibility_info(version)
    print(f"  {version:15s} -> {info.status}")
    if info.guidance:
        print(f"                    {info.guidance[:80]}")

  44.0.0          -> supported
                    Minimum supported version.
  44.2.2          -> supported
                    Latest validated release.
  45.0.0          -> untested
                    Version 45.0.0 is not listed in the compatibility matrix. It may work but has no
  43.0.0          -> unsupported
                    Version 43.0.0 is unsupported. Minimum supported version is 44.0.0. See docs/com
  not-installed   -> unsupported
                    Version not-installed is unsupported. Minimum supported version is 44.0.0. See d


---
## What We Built (Summary)

All of the above is the **foundation layer** — the plumbing that everything else sits on top of.

| Capability | In plain terms |
|---|---|
| Data loading | Read CSV/Parquet files with automatic validation — bad files are rejected with clear error messages |
| Provenance tracking | Every dataset gets a fingerprint and an ID card so we can always trace where results came from |
| Emission factor lookup | Look up how much CO2 is emitted per km driven, per kWh of gas, etc. — by category and year |
| Column name mapping | Translate between OpenFisca's naming and ours, configured in one YAML file |
| Tax calculator adapter | A plug-and-play interface to OpenFisca — can read pre-computed files, call the live API, or use a fake for testing |
| Quality checks | Automatic validation of results: no missing values, correct types, values in sensible ranges |
| Version compatibility | Know at a glance whether a given OpenFisca version is supported |

**What comes next:** Scenario Templates (Epic 2) will use this foundation to define reusable policy scenarios — e.g. "carbon tax at 44.6/tCO2 with lump-sum redistribution" — and the Dynamic Orchestrator (Epic 3) will run them across multiple years.

In [17]:
# Cleanup
import shutil
shutil.rmtree(adapter_dir, ignore_errors=True)